In [2]:
import pandas as pd
import numpy as np
from xgboost import XGBClassifier
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import accuracy_score, log_loss

print("imports work")

imports work


In [3]:
df = pd.read_csv("../data/f1_features.csv")
print(df.shape)
print(df.columns.tolist())

(6915, 17)
['season', 'round', 'circuit', 'driver', 'team', 'grid', 'position', 'points', 'won', 'driver_avg_finish_last3', 'driver_circuit_win_rate', 'team_avg_points_last3', 'driver_dnf_rate', 'regulation_era', 'driver_championship_pos', 'home_race', 'gap_to_leader']


In [4]:
FEATURES = [
    'grid',
    'driver_avg_finish_last3',
    'driver_circuit_win_rate',
    'team_avg_points_last3',
    'driver_dnf_rate',
    'regulation_era',
    'driver_championship_pos',
    'home_race',
    'gap_to_leader'
]

TARGET = "won"

In [21]:
train = df[df["season"] <= 2024]
test = df[df["season"] == 2025]

X_train = train[FEATURES]
y_train = train[TARGET]
X_test = test[FEATURES]
y_test = test[TARGET]

print(f"Training rows: {len(X_train)}")
print(f"Test rows: {len(X_test)}")
print(f"Win rate in train: {y_train.mean():.3f}")
print(f"Win rate in test: {y_test.mean():.3f}")

Training rows: 6436
Test rows: 479
Win rate in train: 0.047
Win rate in test: 0.050


In [23]:
model = XGBClassifier(
    n_estimators=100,
    max_depth=3,
    learning_rate=0.1,
    random_state=42,
    eval_metric='logloss'
)

model.fit(X_train, y_train)

y_pred_proba = model.predict_proba(X_test)[:, 1]
y_pred = (y_pred_proba >= 0.5).astype(int)

accuracy = accuracy_score(y_test, y_pred)
logloss = log_loss(y_test, y_pred_proba)

print(f"Test Accuracy: {accuracy:.3f}")
print(f"Test Log Loss: {logloss:.3f}")

Test Accuracy: 0.956
Test Log Loss: 0.088


In [24]:
# Baseline model that always predicts 0 (no win)
baseline_pred = np.zeros(len(y_test))

baseline_accuracy = accuracy_score(y_test, baseline_pred)
baseline_logloss = log_loss(y_test, np.full(len(y_test), 0.05))

print(f"Baseline Accuracy: {baseline_accuracy:.3f}")
print(f"Baseline Log Loss: {baseline_logloss:.3f}")
print()
print(f"Model Accuracy: {accuracy:.3f}")
print(f"Model Log Loss: {logloss:.3f}")

Baseline Accuracy: 0.950
Baseline Log Loss: 0.199

Model Accuracy: 0.956
Model Log Loss: 0.088


In [25]:
# CHeck model predictions for 2023 Monaco GP
monaco_2023 = test[test["circuit"] == "monaco"].copy()
monaco_2023["win_probability"] = y_pred_proba[monaco_2023.index - test.index[0]]

print(monaco_2023[["driver", "grid", "win_probability"]]
      .sort_values("win_probability", ascending=False)
      .head(10))

              driver  grid  win_probability
6576          norris     1         0.382545
6577         leclerc     2         0.159931
6578         piastri     3         0.150541
6579  max_verstappen     4         0.050567
6584           albon    10         0.013984
6581          hadjar     5         0.008509
6586         russell    14         0.003776
6580        hamilton     7         0.003662
6594          alonso     6         0.002584
6583          lawson     9         0.002479


In [26]:
# Save trianed model
import joblib
import os

os.makedirs("../models", exist_ok=True)

joblib.dump(model, "../models/f1_model.pkl")
print("Model saved!")

Model saved!
